# A model that remembers you

This notebook answers a common question: **does the model know me and remember our last chats?**

Short answer: the model itself does **not** remember you between separate requests. It only sees the `messages` you send each time.

Then we will build a simple memory system in the browser. The memory lives in a small **SQLite database** (a small store of tables you can search). By default it lives only in this browser tab, so it starts empty each time you reload the page.

## Step 1: Get set up

Small cells that bring in the Python tools we need, check that your browser can run models, load the WebLLM library, and set up the model loader.

### 1a: Import tools

- `json` turns Python data into text and back. We use it to hand data to JavaScript.
- `math` has square roots and other math functions.
- `sqlite3` creates and searches a small **database** (an organized store of data you can search).
- `datetime` gives the current date and time.
- `pyjs` connects Python to the browser's **JavaScript**, the language web pages run on. WebLLM is written in JavaScript, so we need this bridge.
- `display` lets a cell show something and then update it in place (we use it for a progress bar).

In [ ]:
import json
import math
import sqlite3
from datetime import datetime, UTC
import pyjs
from IPython.display import display

js = pyjs.js  # a doorway from Python into the browser's JavaScript

print("Tools imported.")

### 1b: Check for WebGPU

Models run on your computer's graphics chip (GPU). Browsers reach the GPU through a feature called **WebGPU**. Recent Chrome and Edge on a laptop have it; many phones and older browsers do not.

In [ ]:
if getattr(js.navigator, "gpu", None) is not None:
    print("WebGPU is available. You can run models.")
else:
    print("WebGPU is not available here. Try the latest Chrome or Edge on a laptop.")

### 1c: Load WebLLM

**WebLLM** is a free JavaScript library that runs language models inside the browser. This line fetches it from the internet (it takes a few seconds the first time).

In [ ]:
webllm = await js.eval("import('https://esm.run/@mlc-ai/web-llm')")
print("WebLLM loaded.")

### 1d: Translate between Python and JavaScript

WebLLM runs in JavaScript, and JavaScript cannot read Python dictionaries directly. Both languages understand **JSON**, a plain-text format for data, so we translate by way of JSON text:

- `js_obj()` turns a Python dictionary into a JavaScript object, for requests we send to WebLLM.
- `py_obj()` goes the other way, turning a JavaScript object (like the model's reply) into Python dictionaries and lists.

You met `js_obj()` in Step 7c of the starter notebook.

In [ ]:
def js_obj(d):
    """Turn a Python dict into a JavaScript object (by way of JSON text)."""
    return js.JSON.parse(json.dumps(d))


def py_obj(value):
    """Turn a JavaScript object into Python dicts and lists (by way of JSON text)."""
    return json.loads(js.JSON.stringify(value))


print("Translators ready.")

### 1e: Set up the model loader

This notebook uses **two** models at once: a chat model and an embedding model. Each one runs in its own **engine** (the part of WebLLM that actually runs a model).

- `chat_engine` and `embed_engine` start as `None` (Python for "nothing yet").
- `progress_bar()` shows a bar that fills up while a model downloads or loads.
- `start_model(model_id, engine)` gets a model ready in an engine and hands the engine back. If the model is not saved in your browser yet, it downloads it first.

This is almost the same loader as in the starter notebook. You do not need to follow every line. Just run it.

In [ ]:
chat_engine = None   # runs the chat model
embed_engine = None  # runs the embedding model


def progress_bar():
    """Show a progress bar that updates in place."""
    status = display("Starting...", display_id=True)

    def show(report):
        filled = int(report.progress * 30)
        bar = "#" * filled + "-" * (30 - filled)
        status.update(f"[{bar}] {report.progress:.0%}\n{report.text}")

    # JavaScript needs its own handle on the Python function to call it
    js_show, handle = pyjs.create_callable(show)
    return status, js_show, handle


async def start_model(model_id, engine):
    """Load a model into an engine (making the engine if needed) and return the engine."""
    status, progress, handle = progress_bar()

    if engine is None:
        engine = webllm.MLCEngine.new()   # create an empty engine
    engine.setInitProgressCallback(progress)

    try:
        await engine.reload(model_id)
    except Exception as error:
        # Some models have two conflicting memory-window settings.
        # If so, turn off the "sliding window" one and try again.
        if "WindowSizeConfigurationError" in str(error):
            status.update("Adjusting this model's settings and retrying...")
            await engine.reload(model_id, js_obj({"sliding_window_size": -1}))
        else:
            raise
    return engine


print("Model loader is ready.")

## Step 2: Load the chat model

We use one chat model for conversation and, later in Step 10, one embedding model for searching memories by meaning. If the chat model is not saved in your browser yet, it downloads first.

In [ ]:
CHAT_MODEL = "gemma3-1b-it-q4f16_1-MLC"

chat_engine = await start_model(CHAT_MODEL, chat_engine)
print(f"{CHAT_MODEL} is loaded and ready.")

## Step 3: A helper for chatting

`chat()` sends a list of messages to the chat model with our usual settings (`max_tokens` and `frequency_penalty` keep small models from rambling or repeating) and returns the answer text. You built the same thing step by step in notebooks 00 and 02.

In [ ]:
async def chat(messages):
    """Send messages to the chat model and return the answer text."""
    request = {
        "messages": messages,
        "temperature": 0.7,
        "max_tokens": 300,
        "frequency_penalty": 0.5,
    }
    reply = py_obj(await chat_engine.chat.completions.create(js_obj(request)))
    return reply["choices"][0]["message"]["content"]


print(await chat([{"role": "user", "content": "Say hello in five words."}]))

## Step 4: Show that the model does not remember across separate requests

In the first request, we tell the model a name. In the second request, we ask for the name again **without** sending the earlier message. If the model answers correctly, that is luck or guessing, not memory.

In [ ]:
first_request = [
    {"role": "user", "content": "My name is Priya. Please say hello and remember it for later."}
]
second_request = [
    {"role": "user", "content": "What is my name? If you do not know, say that clearly."}
]

first_answer = await chat(first_request)
second_answer = await chat(second_request)

print("First reply:")
print(first_answer)
print()
print("Second reply, in a fresh request:")
print(second_answer)

## Step 5: Show that the model can use what is inside `messages`

Now we send the earlier user message **and** the assistant's reply back to the model, then ask again. This is the ordinary way chat apps create the feeling of memory: they resend the conversation every time.

In [ ]:
with_history = [
    first_request[0],
    {"role": "assistant", "content": first_answer},
    {"role": "user", "content": "What is my name?"},
]

print(await chat(with_history))

## Step 6: Create a memory database

We store each exchange in a small SQLite database.

### 6a: Choose where the memory lives

`SAVE_TO_FILE` picks where the database is kept:

- `False` (the default): in the computer's memory, using SQLite's special name `":memory:"`. This is fast and reliable, but the memory is wiped when you reload the page or restart the kernel.
- `True`: in a file called `chat_memory.sqlite3`, saved in this site's storage in **this browser only**. The memory survives a reload, but browser storage is not as sturdy as a normal hard drive, so saving can sometimes fail.

The cell also sets up three helpers:

- `open_memory_database()` opens the database and creates the `conversations` table if it is new. If a saved file cannot be opened, it says so and uses memory instead.
- `execute_write()` saves data, and `fetch_rows()` reads it. If the saved file fails, they print a friendly message instead of an error.

Real chat apps keep memory in a database on a server, which is why your memories follow you between devices there.

In [ ]:
SAVE_TO_FILE = False   # True keeps memories after a reload, but is less reliable in the browser

DB_PATH = "chat_memory.sqlite3" if SAVE_TO_FILE else ":memory:"
FILE_PROBLEM = "The saved memory file did not work. Set SAVE_TO_FILE = False in 6a and run Step 6 again."


def utc_now_text():
    return datetime.now(UTC).isoformat(timespec="seconds")


def open_memory_database(path):
    try:
        connection = sqlite3.connect(path)
        connection.execute("""
        CREATE TABLE IF NOT EXISTS conversations (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            created_at TEXT NOT NULL,
            label TEXT NOT NULL,
            user_text TEXT NOT NULL,
            assistant_text TEXT NOT NULL,
            embedding_json TEXT
        )
        """)
        connection.commit()
        return connection
    except sqlite3.DatabaseError:
        print("The saved memory file could not be opened, so this run keeps memory in the computer's memory instead.")
        return open_memory_database(":memory:")


def execute_write(sql, params):
    try:
        conn.execute(sql, params)
        conn.commit()
    except sqlite3.DatabaseError:
        print(FILE_PROBLEM)


def fetch_rows(sql, params=()):
    try:
        return conn.execute(sql, params).fetchall()
    except sqlite3.DatabaseError:
        print(FILE_PROBLEM)
        return []


print("Database helpers are ready. Memory will be kept in: " + DB_PATH)

### 6b: Open the database

The `conversations` table has one row per exchange: when it happened, a label you choose, what you said, what the model said, and (later) an embedding.

In [ ]:
if "conn" in globals() and conn is not None:
    conn.close()   # close an older connection if this cell is run again

conn = open_memory_database(DB_PATH)
print("Database ready.")

## Step 7: Save chats into the database

### 7a: The saving helpers

- `save_memory()` adds one row to the `conversations` table. The `?` marks in the SQL are filled in, in order, by the values that follow, which is the safe way to put text into a database.
- `chat_and_store()` sends one message to the model, saves both sides of the exchange, and returns the reply.

In [ ]:
def save_memory(label, user_text, assistant_text, embedding=None):
    execute_write(
        "INSERT INTO conversations (created_at, label, user_text, assistant_text, embedding_json) VALUES (?, ?, ?, ?, ?)",
        (
            utc_now_text(),
            label,
            user_text,
            assistant_text,
            json.dumps(embedding) if embedding is not None else None,
        ),
    )


async def chat_and_store(user_text, label="general"):
    assistant_text = await chat([{"role": "user", "content": user_text}])
    save_memory(label, user_text, assistant_text)
    return assistant_text


print("Saving helpers are ready.")

### 7b: Save a few chats

Change `USER_TEXT` and run this cell several times to save a few short chats. Use `MEMORY_LABEL` to group them, like `school`, `hobbies`, or `travel`.

In [ ]:
MEMORY_LABEL = "school"
USER_TEXT = "I am studying biology and I like flash cards. Give me one study tip."

print(await chat_and_store(USER_TEXT, label=MEMORY_LABEL))

## Step 8: Look at the saved memory

This reads the five newest saved chats back out of the database with a SQL `SELECT` query.

In [ ]:
rows = fetch_rows(
    "SELECT id, created_at, label, user_text, assistant_text FROM conversations ORDER BY id DESC LIMIT 5"
)

if not rows:
    print("No saved chats yet. Run Step 7b first.")
for row_id, created_at, label, user_text, assistant_text in rows:
    print(f"#{row_id} | {created_at} | {label}")
    print("  user:      " + user_text)
    print("  assistant: " + assistant_text)
    print()

## Step 9: Use the last few chats as memory

A simple memory system: take the last few saved exchanges, turn them back into `user` and `assistant` messages, and put them in front of the new question. `RECENT` sets how many exchanges to include.

In [ ]:
RECENT = 3
NEW_QUESTION = "Based on what you know about me, suggest a better study plan."


def load_recent_memory(limit):
    rows = fetch_rows(
        "SELECT user_text, assistant_text FROM conversations ORDER BY id DESC LIMIT ?",
        (limit,),
    )
    rows.reverse()   # oldest first, like a real conversation
    messages = []
    for user_text, assistant_text in rows:
        messages.append({"role": "user", "content": user_text})
        messages.append({"role": "assistant", "content": assistant_text})
    return messages


messages = load_recent_memory(RECENT) + [{"role": "user", "content": NEW_QUESTION}]
print(f"Sending {len(messages)} messages.\n")
print(await chat(messages))

## Step 10: Optional: memory by meaning, with embeddings

Looking up only the last few chats is simple, but sometimes an older chat is more relevant. We can search by **meaning** instead of only by time.

An **embedding** is a list of numbers that tries to capture the meaning of a piece of text. Similar texts should end up with similar embeddings. This part is optional because it needs a second model.

### 10a: Load the embedding model

It runs in its own engine, so the chat model stays loaded too. It may take a little while to download the first time.

In [ ]:
EMBED_MODEL = "snowflake-arctic-embed-m-q0f32-MLC-b4"

embed_engine = await start_model(EMBED_MODEL, embed_engine)
print(f"{EMBED_MODEL} is loaded and ready.")

### 10b: Embedding helpers

- `embed_texts()` sends a list of texts to the embedding model and returns one vector (list of numbers) for each.
- `combined_text()` joins both sides of a saved chat into one text. This model expects text wrapped in the markers `[CLS]` (start) and `[SEP]` (end).
- `cosine_similarity()` scores how similar two vectors are: near 1 means very similar (notebook 03 explains it).

In [ ]:
async def embed_texts(texts):
    reply = py_obj(await embed_engine.embeddings.create(js_obj({"input": texts})))
    return [item["embedding"] for item in reply["data"]]


def combined_text(user_text, assistant_text):
    return "[CLS] " + user_text + " \n " + assistant_text + " [SEP]"


def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


print("Embedding helpers are ready.")

## Step 11: Add embeddings to saved chats

This finds saved chats that do not have an embedding yet, makes one for each, and stores it in the `embedding_json` column. `json.dumps` turns the list of numbers into text so the database can store it.

In [ ]:
rows = fetch_rows(
    "SELECT id, user_text, assistant_text FROM conversations WHERE embedding_json IS NULL ORDER BY id"
)

if not rows:
    print("All saved chats already have embeddings, or there are no chats yet.")
else:
    texts = [combined_text(user_text, assistant_text) for _, user_text, assistant_text in rows]
    embeddings = await embed_texts(texts)
    for (row_id, _, _), embedding in zip(rows, embeddings):
        execute_write(
            "UPDATE conversations SET embedding_json = ? WHERE id = ?",
            (json.dumps(embedding), row_id),
        )
    print(f"Added embeddings to {len(rows)} saved chats.")

## Step 12: Find the most relevant past chats by meaning

Now we embed the new question, score every saved memory against it with cosine similarity, and keep the `TOP_K` best matches.

This embedding model was trained to find matching passages when a search question starts with the words "Represent this sentence for searching relevant passages:", so we add them to the question.

In [ ]:
QUERY = "What study method fits someone who likes flash cards and biology?"
TOP_K = 2

rows = fetch_rows(
    "SELECT id, label, user_text, assistant_text, embedding_json FROM conversations WHERE embedding_json IS NOT NULL"
)

top_rows = []
if not rows:
    print("No embedded memories yet. Run Steps 7b, 10 and 11 first.")
else:
    query_text = "[CLS] Represent this sentence for searching relevant passages: " + QUERY + " [SEP]"
    query_embedding = (await embed_texts([query_text]))[0]

    scored = []
    for row in rows:
        score = cosine_similarity(query_embedding, json.loads(row[4]))
        scored.append((score, row))
    scored.sort(reverse=True, key=lambda item: item[0])
    top_rows = scored[:TOP_K]

    print("Most similar saved chats:")
    for score, (row_id, label, user_text, assistant_text, _) in top_rows:
        print(f"score={score:.3f} | #{row_id} | {label}")
        print("  user:      " + user_text)
        print("  assistant: " + assistant_text)
        print()

## Step 13: Use relevant memories in a new answer

The final step turns the matching memories back into `messages`, adds the new question, and asks the chat model.

In [ ]:
if not top_rows:
    print("No matching memories yet. Run Step 12 first.")
else:
    memory_messages = []
    for score, (_, _, user_text, assistant_text, _) in top_rows:
        memory_messages.append({"role": "user", "content": user_text})
        memory_messages.append({"role": "assistant", "content": assistant_text})
    memory_messages.append({"role": "user", "content": QUERY})
    print(await chat(memory_messages))

## Your turn

Add a few more facts about yourself in Step 7b, then try both memory methods: the last few chats (Step 9) and memory by meaning (Steps 11 to 13). Which gives better memory for your questions? Remember that all of this memory lives only in this browser. Try `SAVE_TO_FILE = True` in Step 6a if you want it to survive a reload.